In [1]:
import tensorflow as tf
import os
import numpy as np
# from osgeo import gdal, osr
# import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F


2026-03-12 13:10:33.706622: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/glade/derecho/scratch/lizhili/CFAT/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
import torch
from torch import nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.conv(x)
        return x


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(Up, self).__init__()
        self.up_scale = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)

    def forward(self, x1, x2):
        x2 = self.up_scale(x2)

        diffY = x1.size()[2] - x2.size()[2]
        diffX = x1.size()[3] - x2.size()[3]

        x2 = F.pad(x2, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return x


class DownLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DownLayer, self).__init__()
        self.pool = nn.MaxPool2d(2, stride=2, padding=0)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        x = self.conv(self.pool(x))
        return x


class UpLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(UpLayer, self).__init__()
        self.up = Up(in_ch, out_ch)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        a = self.up(x1, x2)
        x = self.conv(a)
        return x


class UNet(nn.Module):
    def __init__(self, input_dims = 4, output_dims=2):
        super(UNet, self).__init__()
        self.conv1 = DoubleConv(input_dims, 64)
        self.down1 = DownLayer(64, 128)
        self.down2 = DownLayer(128, 256)
        self.down3 = DownLayer(256, 512)
        self.down4 = DownLayer(512, 1024)
        self.up1 = UpLayer(1024, 512)
        self.up2 = UpLayer(512, 256)
        self.up3 = UpLayer(256, 128)
        self.up4 = UpLayer(128, 64)
        self.last_conv = nn.Conv2d(64, output_dims, 1)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x1_up = self.up1(x4, x5)
        x2_up = self.up2(x3, x1_up)
        x3_up = self.up3(x2, x2_up)
        x4_up = self.up4(x1, x3_up)
        output = self.last_conv(x4_up)
        return output

In [4]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1408
num_training = 1126
num_test = num_sample-num_training
dataset_name = 'M2L8_CHM'

In [5]:
file_folder = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()
output_tfrecords_files

['/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ATD_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_BiDiff_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CAMixer_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CFAT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ESRGAN_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_RGT_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_SED_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_SRNO_x16.tfrecords',
 '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_UPSR_x16.tfrecords']

In [6]:
file_folder = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords'
output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]
output_tfrecords_files.sort()

for output_tfrecords in output_tfrecords_files:
    dirname = '/glade/derecho/scratch/lizhili/m2l8/unet_pths_16x/'  # one level up (/content/drive/MyDrive/GeoSR)
    filename = os.path.basename(output_tfrecords)                 # e.g. M2L8_River_SRNO_x4.tfrecords
    base = filename.replace(f"{dataset_name}_", "").replace("_x16.tfrecords", "")
    segformer_save_path =  os.path.join(dirname, f"{dataset_name}_Unet_{base}_run2.pth")
    print(output_tfrecords, " Save to:", segformer_save_path)

/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ATD_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/m2l8/unet_pths_16x/M2L8_CHM_Unet_ATD_run2.pth
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_BiDiff_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/m2l8/unet_pths_16x/M2L8_CHM_Unet_BiDiff_run2.pth
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CAMixer_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/m2l8/unet_pths_16x/M2L8_CHM_Unet_CAMixer_run2.pth
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CFAT_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/m2l8/unet_pths_16x/M2L8_CHM_Unet_CFAT_run2.pth
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ESRGAN_x16.tfrecords  Save to: /glade/derecho/scratch/lizhili/m2l8/unet_pths_16x/M2L8_CHM_Unet_ESRGAN_run2.pth
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_RGT_x16.tfrecords  Save to: /glad

In [7]:
def input_pipeline_downstream(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):

    feature_description = {
        'hres': tf.io.FixedLenFeature([7*hres_size_4x*hres_size_4x], dtype=tf.int64),
        'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
    }
    
    @tf.function
    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)

        hres = feature_dict['hres']
        hres = tf.reshape(hres, [7, hres_size_4x, hres_size_4x])
        hres = tf.cast(hres, tf.float32)
        hres = hres*0.0000275-0.2

        label = feature_dict['label']
        label = tf.reshape(label, [label_size, label_size, 1])
        label = tf.cast(label, tf.float32)*0.1

        return hres, label[..., 0]
        
    @tf.function
    def _augment_function(hres_img, label):
        # Transpose to [H, W, C]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
        if tf.rank(label) == 2:
            label = tf.expand_dims(label, axis=-1)
    
        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
        hres_img = tf.image.rot90(hres_img, k=k)
        label = tf.image.rot90(label, k=k)

        # ---- Random horizontal flip ----
        do_flip_lr = tf.random.uniform([]) > 0.5
        hres_img = tf.cond(do_flip_lr,
                           lambda: tf.image.flip_left_right(hres_img),
                           lambda: hres_img)
        label = tf.cond(do_flip_lr,
                        lambda: tf.image.flip_left_right(label),
                        lambda: label)
    
        # ---- Random vertical flip ----
        do_flip_ud = tf.random.uniform([]) > 0.5
        hres_img = tf.cond(do_flip_ud,
                           lambda: tf.image.flip_up_down(hres_img),
                           lambda: hres_img)
        label = tf.cond(do_flip_ud,
                        lambda: tf.image.flip_up_down(label),
                        lambda: label)
    
        # Transpose back to [C, H, W]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
        label = tf.squeeze(label, axis=-1)
    
        return hres_img, label

    dataset = tf.data.TFRecordDataset(filename)
    dataset = dataset.skip(skip)
    if take:
        dataset = dataset.take(take)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)

    return batch

# filenames = '/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_Urban_ATD_x16.tfrecords'
# ds = input_pipeline_downstream(filenames, 10, 0, 200, is_shuffle=True, is_train=True, is_repeat=True)


# for lres_batch, hres_batch in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))
#         hres_img = hres_batch[i].numpy()

#         axes[0].imshow(lres_img[:, :, 3:0:-1]*3)
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img)
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         plt.show()

#     break

In [8]:
model_order = ["ATD", "RGT", "CFAT", "CAMixer", "SRNO", "BiDiff", "SED", "ESRGAN", "UPSR"]

order_map = {m: i for i, m in enumerate(model_order)}

def extract_model_name(path):
    # ChesapeakeRSC_<MODEL>_x16.tfrecords
    return path.split('_')[-2]

# Sort files by model order
output_tfrecords_files = sorted(
    output_tfrecords_files,
    key=lambda x: order_map.get(extract_model_name(x), float('inf'))
)

for f in output_tfrecords_files:
    print(f)

/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ATD_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_RGT_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CFAT_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_CAMixer_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_SRNO_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_BiDiff_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_SED_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_ESRGAN_x16.tfrecords
/glade/derecho/scratch/lizhili/m2l8/x16_downstream_tfrecords/M2L8_CHM_UPSR_x16.tfrecords


In [ ]:
# file_folder = '/glade/derecho/scratch/lizhili/s2naip/x16_downstream_datasets'
# output_tfrecords_files = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith(dataset_name) and f.endswith('tfrecords')]

# storage: {row_name: {model_name: value}}
mf1_results = {}
acc_results = {}

def extract_model_name(filename):
    # e.g. ChesapeakeRSC_SRNO_x16.tfrecords → SRNO
    return filename.split('_')[-2]

def make_row_name(dataset_name, run_id):
    return f"{dataset_name}_run{run_id}"

for run_id in range(1,4):
    for output_tfrecords in output_tfrecords_files:
        filename = os.path.basename(output_tfrecords)
        model_name = extract_model_name(filename)
        print(model_name)
        row_name = make_row_name(dataset_name, run_id)
        print(row_name)
        
        dirname = '/glade/derecho/scratch/lizhili/m2l8/unet_pths_16x/'  # one level up (/content/drive/MyDrive/GeoSR)
        filename = os.path.basename(output_tfrecords)                 # e.g. M2L8_River_SRNO_x4.tfrecords
        base = filename.replace(f"{dataset_name}_", "").replace("_x16.tfrecords", "")
        segformer_save_path =  os.path.join(dirname, f"{dataset_name}_Unet_{base}_run{run_id}.pth")
        print(output_tfrecords)
        print('Load: ', segformer_save_path)
    
        print('Begin Segformer Training')
        device = 'cuda'
        model_naip = UNet(input_dims=7, output_dims=class_num).to(device)
        model_naip.load_state_dict(torch.load(segformer_save_path.replace(".pth", "_best.pth"), weights_only=True))
        model_naip.eval()
        print('load successful!')
    
        print('Begin Testing')
        test_ds = input_pipeline_downstream(output_tfrecords, 4, num_training, num_test, is_shuffle=False, is_train=False, is_repeat=False)
    
        mae_total = 0.0
        rmse_total = 0.0
        pixel_count = 0
    
        for images, labels in test_ds:
            images = torch.from_numpy(images.numpy().astype('float32')).to(device)
    
            with torch.no_grad():
                preds = model_naip(images)
                preds = F.interpolate(preds, size=(label_size, label_size), mode='nearest')
                preds = preds.squeeze(1).cpu().numpy()  # [B, H, W] for regression
    
            labels = labels.numpy()  # [B, H, W] if channel dim exists
    
            abs_error = np.abs(preds - labels)
            sq_error = (preds - labels) ** 2
    
            mae_total += np.sum(abs_error)
            rmse_total += np.sum(sq_error)
            pixel_count += np.prod(labels.shape)
    
        mae = mae_total / pixel_count
        rmse = np.sqrt(rmse_total / pixel_count)
    
        print("MAE:", round(mae, 4))
        print("RMSE:", round(rmse, 4))
    
        # ----- store -----
        mf1_results.setdefault(row_name, {})[model_name] = mae
        acc_results.setdefault(row_name, {})[model_name] = rmse
    


In [10]:
import pandas as pd

mf1_df = pd.DataFrame.from_dict(mf1_results, orient='index').sort_index()
acc_df = pd.DataFrame.from_dict(acc_results, orient='index').sort_index()

# Add average row (mean over rows, i.e., runs/datasets)
mf1_df.loc["Average"] = mf1_df.mean(axis=0)
acc_df.loc["Average"] = acc_df.mean(axis=0)

mf1_df.to_excel(f"{dataset_name}_Unet_mae_UPSR.xlsx")
acc_df.to_excel(f"{dataset_name}_Unet_rmse_UPSR.xlsx")

print("Saved mf1.xlsx and accuracy.xlsx")

Saved mf1.xlsx and accuracy.xlsx


In [11]:
mf1_df

,UPSR
M2L8_CHM_run1,0.600301
M2L8_CHM_run2,0.613804
M2L8_CHM_run3,0.593774
Average,0.602626
